<a href="https://colab.research.google.com/github/elvisolickal/RAG---Internship/blob/main/RAG_PHASE_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain chromadb pypdf langchain-google-genai langchain-groq langgraph langchain-community langchain-text-splitters google-cloud-aiplatform langchain-google-vertexai -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from google.colab import userdata
gemini_api_key=userdata.get('GEMINI_KEY')

# 1. Set your API key for Google Gemini (used for embeddings)
os.environ["GOOGLE_API_KEY"] = gemini_api_key

/tmp/ipykernel_1396/3568705262.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [20]:
pdf_files = [


    "/content/Computer Vision.pdf",
    "/content/Generative AI and Large Language Models.pdf",
    "/content/History of AI.pdf",
    "/content/Machine Learning Fundamentals.pdf",
    "/content/Natural Language Processing.pdf",
    "/content/Neural Networks.pdf",
    "/content/Reinforcement Learning.pdf",
    "/content/Supervised Learning.pdf",

]
PERSIST_DIR = "./chroma_db"

In [21]:
# 2. Safe File Loading 32,135 words 195,154 characters
all_docs = []

for file_path in pdf_files:
    print(f"Attempting to load: {file_path}...")
    try:
        loader = PyPDFLoader(file_path)
        docs = loader.load()
        if not docs:
            print(f"Warning: {file_path} was loaded, but no text was found.")
        else:
            all_docs.extend(docs)
            print(f"Successfully loaded {len(docs)} pages from this part.")
    except Exception as e:
        print(f"Error loading the PDF: {e}")

print(f"\nDone! Combined total of {len(all_docs)} pages loaded into memory.")

Attempting to load: /content/Computer Vision.pdf...
Successfully loaded 16 pages from this part.
Attempting to load: /content/Generative AI and Large Language Models.pdf...
Successfully loaded 11 pages from this part.
Attempting to load: /content/History of AI.pdf...
Successfully loaded 8 pages from this part.
Attempting to load: /content/Machine Learning Fundamentals.pdf...
Successfully loaded 18 pages from this part.
Attempting to load: /content/Natural Language Processing.pdf...
Successfully loaded 18 pages from this part.
Attempting to load: /content/Neural Networks.pdf...
Successfully loaded 17 pages from this part.
Attempting to load: /content/Reinforcement Learning.pdf...
Successfully loaded 14 pages from this part.
Attempting to load: /content/Supervised Learning.pdf...
Successfully loaded 19 pages from this part.

Done! Combined total of 121 pages loaded into memory.


In [22]:
import re

# ==========================================
# 3. Sentence-wise Chunking with Text Cleaning
# ==========================================

try:
    print("Splitting text into clean sentence-based chunks...")

    # --------------------------------------------------
    # Configuration Parameters
    # --------------------------------------------------

    # Number of sentences to include in each chunk
    SENTENCES_PER_CHUNK = 5

    # Number of overlapping sentences between consecutive chunks
    OVERLAP = 2

    # Validate configuration
    if OVERLAP >= SENTENCES_PER_CHUNK:
        raise ValueError(
            "OVERLAP must be smaller than SENTENCES_PER_CHUNK."
        )

    # List to store all generated chunks
    chunks = []

    # Number of sentences to move forward after every chunk
    step = SENTENCES_PER_CHUNK - OVERLAP

    # --------------------------------------------------
    # Process every document/page
    # --------------------------------------------------

    for doc in all_docs:

        try:

            # ------------------------------------------
            # Extract page text
            # ------------------------------------------

            text = doc.page_content

            # ------------------------------------------
            # Clean common PDF extraction artifacts
            # ------------------------------------------

            # Replace newlines, carriage returns and tabs with spaces
            text = text.replace("\n", " ")
            text = text.replace("\r", " ")
            text = text.replace("\t", " ")

            # Collapse multiple spaces into a single space
            text = " ".join(text.split())

            # Remove unwanted spaces before punctuation
            text = re.sub(r"\s+([.,!?;:])", r"\1", text)

            # Remove leading and trailing whitespace
            text = text.strip()

            # ------------------------------------------
            # Skip completely empty pages
            # ------------------------------------------

            if not text:
                print(
                    f"Warning: Empty page found in "
                    f"{doc.metadata.get('source', 'Unknown File')} "
                    f"(Page {doc.metadata.get('page')})"
                )
                continue

            # ------------------------------------------
            # Split text into sentences
            # ------------------------------------------
            # Splits after '.', '!' or '?'
            # while preserving punctuation.
            # Example:
            # "AI is useful. ML is powerful!"
            # ->
            # ["AI is useful.", "ML is powerful!"]
            # ------------------------------------------

            sentences = re.split(r'(?<=[.!?])\s+', text)

            # Remove empty sentences
            sentences = [
                sentence.strip()
                for sentence in sentences
                if sentence.strip()
            ]

            # ------------------------------------------
            # Handle pages where sentence detection fails
            # ------------------------------------------

            if len(sentences) == 0:

                print(
                    f"Warning: No valid sentences found in "
                    f"{doc.metadata.get('source', 'Unknown File')} "
                    f"(Page {doc.metadata.get('page')})"
                )

                # Store the cleaned page as one chunk instead
                chunk = doc.__class__(
                    page_content=text,
                    metadata=doc.metadata
                )

                chunks.append(chunk)

                continue

            # ------------------------------------------
            # Create overlapping sentence chunks
            # ------------------------------------------

            for i in range(0, len(sentences), step):

                # Select the current group of sentences
                current_sentences = sentences[
                    i:i + SENTENCES_PER_CHUNK
                ]

                # Skip empty chunks
                if not current_sentences:
                    continue

                # Combine sentences into a single chunk
                chunk_text = " ".join(current_sentences).strip()

                # Create a new Document object while preserving metadata
                chunk = doc.__class__(
                    page_content=chunk_text,
                    metadata=doc.metadata
                )

                # Add chunk to the master list
                chunks.append(chunk)

        # ------------------------------------------
        # Handle errors for individual documents
        # ------------------------------------------

        except Exception as doc_error:

            print(
                f"Error processing "
                f"{doc.metadata.get('source', 'Unknown File')} "
                f"(Page {doc.metadata.get('page')}): "
                f"{doc_error}"
            )

            # Continue processing the remaining documents
            continue

    # --------------------------------------------------
    # Final Summary
    # --------------------------------------------------

    print("\nChunking completed successfully.")
    print(f"Total chunks created: {len(chunks)}")

# --------------------------------------------------
# Configuration Errors
# --------------------------------------------------

except ValueError as config_error:

    print(f"Configuration Error: {config_error}")

# --------------------------------------------------
# Unexpected Errors
# --------------------------------------------------

except Exception as e:

    print(f"Unexpected error during chunking: {e}")

    exit()

Splitting text into clean sentence-based chunks...

Chunking completed successfully.
Total chunks created: 440


In [23]:
# update the chroma database
import shutil
import os

if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")
    print("Old ChromaDB deleted.")

In [24]:
import time
from langchain_community.vectorstores import Chroma

# 4. Embed & Persist to Disk (Batch-processed to avoid 429 Quota Limits)
try:
    print("Connecting to Gemini API and initializing ChromaDB...")
    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

    # Initialize an empty Chroma vectorstore first
    print("Initializing vector store...")
    vectorstore = Chroma(
        persist_directory=PERSIST_DIR,
        embedding_function=embeddings
    )

    # Define a safe batch size (under the 100 requests/min limit)
    BATCH_SIZE = 50 # Reduced from 90 to stay within 100 req/min limit
    total_chunks = len(chunks)

    print(f"Starting batch ingestion for {total_chunks} chunks...")
    for i in range(0, total_chunks, BATCH_SIZE):
        batch = chunks[i:i + BATCH_SIZE]
        print(f"Embedding chunks {i} to {min(i + BATCH_SIZE, total_chunks)}...")

        # Add this specific batch to the vectorstore
        vectorstore.add_documents(batch)

        # Wait 60 seconds between batches to let the Gemini API quota reset
        if i + BATCH_SIZE < total_chunks:
            print("⏳ Waiting 60 seconds for API quota to reset...")
            time.sleep(60)

    print("All chunks successfully embedded and persisted to disk!")

except Exception as e:
    print(f"Error during embedding or database storage: {e}")

Connecting to Gemini API and initializing ChromaDB...
Initializing vector store...


/tmp/ipykernel_1396/2179478358.py:11: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Starting batch ingestion for 440 chunks...
Embedding chunks 0 to 50...
⏳ Waiting 60 seconds for API quota to reset...
Embedding chunks 50 to 100...
⏳ Waiting 60 seconds for API quota to reset...
Embedding chunks 100 to 150...
⏳ Waiting 60 seconds for API quota to reset...
Embedding chunks 150 to 200...
⏳ Waiting 60 seconds for API quota to reset...
Embedding chunks 200 to 250...
⏳ Waiting 60 seconds for API quota to reset...
Embedding chunks 250 to 300...
⏳ Waiting 60 seconds for API quota to reset...
Embedding chunks 300 to 350...
⏳ Waiting 60 seconds for API quota to reset...
Embedding chunks 350 to 400...
⏳ Waiting 60 seconds for API quota to reset...
Embedding chunks 400 to 440...
All chunks successfully embedded and persisted to disk!


In [25]:
# Zip the chroma_db folder so you can download it
!zip -r chroma_phase4_new_db.zip ./chroma_db

  adding: chroma_db/ (stored 0%)
  adding: chroma_db/chroma.sqlite3 (deflated 22%)
  adding: chroma_db/4b48059f-520c-4eee-ad91-b5dc666b1f86/ (stored 0%)
  adding: chroma_db/4b48059f-520c-4eee-ad91-b5dc666b1f86/length.bin (deflated 67%)
  adding: chroma_db/4b48059f-520c-4eee-ad91-b5dc666b1f86/header.bin (deflated 63%)
  adding: chroma_db/4b48059f-520c-4eee-ad91-b5dc666b1f86/data_level0.bin (deflated 100%)
  adding: chroma_db/4b48059f-520c-4eee-ad91-b5dc666b1f86/link_lists.bin (stored 0%)


In [27]:
from google.colab import files
files.download("chroma_phase4_new_db.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# Unzip the database back into the environment
!unzip chroma_phase4_db.zip

Archive:  chroma_phase4_db.zip
   creating: chroma_db/
  inflating: chroma_db/chroma.sqlite3  
   creating: chroma_db/e9aa41a8-cd3c-4cad-9219-584dae3941bd/
  inflating: chroma_db/e9aa41a8-cd3c-4cad-9219-584dae3941bd/length.bin  
  inflating: chroma_db/e9aa41a8-cd3c-4cad-9219-584dae3941bd/header.bin  
  inflating: chroma_db/e9aa41a8-cd3c-4cad-9219-584dae3941bd/data_level0.bin  
 extracting: chroma_db/e9aa41a8-cd3c-4cad-9219-584dae3941bd/link_lists.bin  


In [28]:
import os

print(os.path.exists("./chroma_db"))
print(os.path.exists("./chroma_db/chroma.sqlite3"))

True
True


In [29]:
import os
from typing import List
from typing_extensions import TypedDict
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langgraph.graph import START, END, StateGraph
from google.colab import userdata
gem_key=userdata.get('GEMINI_KEY')
groq_key=userdata.get('GROQ_KEY')
# ==========================================
# 1. ENVIRONMENT SETUP & INITIALIZATION
# ==========================================
os.environ["GOOGLE_API_KEY"] = gem_key
os.environ["GROQ_API_KEY"] = groq_key

PERSIST_DIR = "./chroma_db"

try:
    # Load the vector store without invoking additional embedding costs
    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
    vectorstore = Chroma(persist_directory=PERSIST_DIR, embedding_function=embeddings)
    # Retrieve exactly 3 chunks to keep your prompt token footprint small
    retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

    # Initialize the Groq LLM
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)
    print("Vector database and Groq LLM initialized successfully.")
except Exception as e:
    print(f"Initialization Error: {e}")
    exit()

# ==========================================
# 2. DEFINE THE GRAPH STATE
# ==========================================
# This object keeps track of all data flowing through our pipeline.
class GraphState(TypedDict):
    question: str       # Input question from the user
    context: List[str]  # Chunks retrieved from ChromaDB
    generation: str    # Final response generated by Groq

# ==========================================
# 3. DEFINE THE NODES (WORKERS)
# ==========================================

def retrieve_node(state: GraphState):
    """
    Takes the question from the state, queries ChromaDB,
    and saves the text chunks back into the state context.
    """
    print("Node 1: Retrieving relevant documents from ChromaDB...")
    try:
        question = state["question"]
        docs = retriever.invoke(question)

        # Extract plain text content from the retrieved document chunks
        context_texts = [doc.page_content for doc in docs]

        # We return an update to the 'context' key in our state
        return {"context": context_texts}
    except Exception as e:
        print(f"Error during retrieval: {e}")
        return {"context": ["Error: Failed to retrieve documentation context."]}


def generate_node(state: GraphState):
    """
    Takes the question and the context chunks, formats the RAG prompt,
    sends it to Groq, and saves the answer in the state.
    """
    print("Node 2: Generating answer using Groq...")
    try:
        question = state["question"]
        context = state["context"]

        # Combine chunks into a single reference block
        formatted_context = "\n\n---\n\n".join(context)

        # Define a precise RAG prompt template
        prompt_template = PromptTemplate(
            template="""You are a helpful academic assistant. Answer the question based ONLY on the provided context. If you do not know the answer or if it is not explicitly mentioned in the context, state that you cannot find the answer in the provided documents.

Context:
{context}

Question: {question}

Answer:""",
            input_variables=["context", "question"]
        )

        # Build and invoke the sequence
        rag_chain = prompt_template | llm
        response = rag_chain.invoke({"context": formatted_context, "question": question})

        return {"generation": response.content}
    except Exception as e:
        print(f"Error during generation: {e}")
        return {"generation": "Error: The generation engine encountered an issue answering this question."}

# ==========================================
# 4. BUILD THE GRAPH FLOW
# ==========================================
# Initialize the StateGraph with our custom State schema
workflow = StateGraph(GraphState)

# Add our processing units (nodes) to the layout
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate", generate_node)

# Connect the nodes via explicit paths (edges)
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", END)

# Compile the workflow into an executable application
app = workflow.compile()
print("LangGraph pipeline successfully compiled.")

Vector database and Groq LLM initialized successfully.
LangGraph pipeline successfully compiled.


In [30]:
# Define your test question
input_data = {"question": "What is Computer Vision"}

# Run the pipeline
try:
    final_state = app.invoke(input_data)

    print("\n" + "="*40)
    print("FINAL RAG RESPONSE:")
    print("="*40)
    print(final_state["generation"])

except Exception as e:
    print(f"Pipeline Execution Failed: {e}")

Node 1: Retrieving relevant documents from ChromaDB...
Node 2: Generating answer using Groq...

FINAL RAG RESPONSE:
Computer Vision is the science of enabling computers to derive meaningful information from digital images, videos, and other visual inputs.


Evaluations

In [36]:
!pip install ragas datasets pandas -q

In [38]:
from importlib.metadata import version

print(version("ragas"))

0.4.3


In [32]:
import sys
import types
import langchain_community.llms

# 1. Create a dummy module to replace the missing VertexAI path
mock_vertex_module = types.ModuleType("langchain_community.chat_models.vertexai")

# 2. Add a hollow ChatVertexAI class to it
mock_vertex_module.ChatVertexAI = type("ChatVertexAI", (object,), {})

# 3. Inject our dummy module directly into Python's system modules
sys.modules["langchain_community.chat_models.vertexai"] = mock_vertex_module

# 4. Patch the legacy llms module as well to prevent the next error on line 13
langchain_community.llms.VertexAI = type("VertexAI", (object,), {})

print("Hotfix applied to system memory. You can now import RAGAS!")

Hotfix applied to system memory. You can now import RAGAS!


In [51]:
import pandas as pd
from datasets import Dataset
from ragas import evaluate

# 1. NEW IMPORTS: Import the capitalized Class names directly
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall

# 2. Import the Ragas wrappers for Langchain objects
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from langchain_groq import ChatGroq
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# --- 1. Define your test questions and their expected answers ---
test_questions = [
    "Who coined the term Artificial Intelligence?",
        "What was the significance of the Dartmouth Conference?",
        "Why did the first AI Winter occur?",
         "What is machine learning?",
        "What is the difference between supervised and unsupervised learning?",
        "What is overfitting in machine learning?",
        "What is backpropagation?",
        "Why are activation functions important in neural networks?",
        "How do CNNs differ from traditional neural networks?",



]

ground_truths = [
     "John McCarthy coined the term Artificial Intelligence during the Dartmouth Conference in 1956.",
"The Dartmouth Conference of 1956 is considered the birth of Artificial Intelligence as a formal field of study.",
"The first AI Winter occurred due to unmet expectations, limited computing power, and reduced funding for AI research.",
"Machine learning is a branch of AI that enables systems to learn patterns from data and improve performance without explicit programming.",
"Supervised learning uses labeled data, while unsupervised learning identifies patterns in unlabeled data.",
"Overfitting occurs when a model learns the training data too closely and performs poorly on unseen data.",
"Backpropagation is a training algorithm that updates neural network weights by propagating errors backward through the network.",
"Activation functions introduce non-linearity into neural networks, enabling them to learn complex patterns.",
"Convolutional Neural Networks use convolution operations to process spatial features and are commonly used in image analysis."

]

# --- 2. Collect the generated answers and context from your LangGraph pipeline ---
data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": ground_truths
}

print("Running 10 questions through the LangGraph pipeline...")
for q in test_questions:
    # Run our LangGraph app from the previous step
    # (Note: Assumes 'app' is defined and initialized earlier in your environment)
    result = app.invoke({"question": q})

    data["question"].append(q)
    data["answer"].append(result["generation"])
    data["contexts"].append(result["context"])


# --- 3. Convert to a HuggingFace Dataset (required by RAGAS) ---
dataset = Dataset.from_dict(data)

# --- 4. Initialize the evaluator models ---
print("Evaluating results with RAGAS (This may take a few minutes)...")
eval_llm = ChatGroq(model="llama-3.1-8b-instant")
eval_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# -> Wrap the Langchain objects so Ragas can process them without the kwargs error
# (Note: You may see a DeprecationWarning for these wrappers, but they are currently
# required to bridge Langchain and Ragas v0.2 smoothly).
ragas_llm = LangchainLLMWrapper(eval_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(eval_embeddings)

# --- 5. Run the evaluation ---
evaluation_result = evaluate(
    dataset=dataset,
    metrics=[
         ContextRecall()  ,       # NEW: Instantiated with parentheses

   # NEW: Instantiated with parentheses
    ],
    llm=ragas_llm,               # Use wrapped LLM
    embeddings=ragas_embeddings  # Use wrapped Embeddings
)

# --- 6. Display the final scorecard ---
df = evaluation_result.to_pandas()
print("\nEVALUATION COMPLETE! Here is your scorecard:")
display(df)

# df.to_csv("rag_evaluation_results.csv", index=False) # Optional: save to file

/tmp/ipykernel_1396/3991520840.py:6: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
/tmp/ipykernel_1396/3991520840.py:6: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall
/tmp/ipykernel_1396/3991520840.py:6: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import Faithfulness, Answer

Running 10 questions through the LangGraph pipeline...
Node 1: Retrieving relevant documents from ChromaDB...
Node 2: Generating answer using Groq...
Node 1: Retrieving relevant documents from ChromaDB...
Node 2: Generating answer using Groq...
Node 1: Retrieving relevant documents from ChromaDB...
Node 2: Generating answer using Groq...
Node 1: Retrieving relevant documents from ChromaDB...
Node 2: Generating answer using Groq...
Node 1: Retrieving relevant documents from ChromaDB...
Node 2: Generating answer using Groq...
Node 1: Retrieving relevant documents from ChromaDB...
Node 2: Generating answer using Groq...
Node 1: Retrieving relevant documents from ChromaDB...
Node 2: Generating answer using Groq...
Node 1: Retrieving relevant documents from ChromaDB...
Node 2: Generating answer using Groq...
Node 1: Retrieving relevant documents from ChromaDB...
Node 2: Generating answer using Groq...
Evaluating results with RAGAS (This may take a few minutes)...


/tmp/ipykernel_1396/3991520840.py:74: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(eval_llm)
/tmp/ipykernel_1396/3991520840.py:75: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(eval_embeddings)


Evaluating:   0%|          | 0/9 [00:00<?, ?it/s]


EVALUATION COMPLETE! Here is your scorecard:


,user_input,retrieved_contexts,response,reference,context_recall
0,Who coined the term Artificial Intelligence?,[The conference was organized by researchers i...,"John McCarthy coined the term ""Artificial Inte...",John McCarthy coined the term Artificial Intel...,1.000
1,What was the significance of the Dartmouth Con...,[The conference was organized by researchers i...,The significance of the Dartmouth Conference i...,The Dartmouth Conference of 1956 is considered...,1.000
2,Why did the first AI Winter occur?,[Many AI systems performed well in controlled ...,The First AI Winter occurred because researche...,The first AI Winter occurred due to unmet expe...,1.000
3,What is machine learning?,[Input Data + Correct Outputs → Learning Algor...,Machine learning is the ability of a model to ...,Machine learning is a branch of AI that enable...,0.000
4,What is the difference between supervised and ...,[Supervised Learning 2. Unsupervised Learning ...,"Based on the provided context, the difference ...","Supervised learning uses labeled data, while u...",1.000
5,What is overfitting in machine learning?,[Overfitting and Underfitting Two common machi...,Overfitting in machine learning occurs when a ...,Overfitting occurs when a model learns the tra...,1.000
6,What is backpropagation?,[Adjust weights accordingly. The error signal ...,Backpropagation is a process where the error s...,Backpropagation is a training algorithm that u...,1.000
7,Why are activation functions important in neur...,[Activation functions introduce non-linearity....,Activation functions are important in neural n...,Activation functions introduce non-linearity i...,0.375
8,How do CNNs differ from traditional neural net...,[They are specifically designed for image proc...,"According to the provided context, CNNs differ...",Convolutional Neural Networks use convolution ...,1.000


In [40]:
faithfulness_df = evaluation_result.to_pandas()
faithfulness_df.to_csv("faithfulness.csv", index=False)

In [45]:
answer_df = evaluation_result.to_pandas()
answer_df.to_csv("answer_relevancy.csv", index=False)

In [50]:
context_precision_df = evaluation_result.to_pandas()
context_precision_df.to_csv("context_precision.csv", index=False)

In [52]:
context_recall_df = evaluation_result.to_pandas()
context_recall_df.to_csv("context_recall.csv", index=False)

In [53]:
import pandas as pd

faithfulness = pd.read_csv("faithfulness.csv")
answer = pd.read_csv("answer_relevancy.csv")
context_precision = pd.read_csv("context_precision.csv")
context_recall = pd.read_csv("context_recall.csv")

# Keep only the metric columns from later runs
final_df = faithfulness.copy()

final_df["answer_relevancy"] = answer["answer_relevancy"]
final_df["context_precision"] = context_precision["context_precision"]
final_df["context_recall"] = context_recall["context_recall"]

display(final_df)

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,Who coined the term Artificial Intelligence?,['The conference was organized by researchers ...,"John McCarthy coined the term ""Artificial Inte...",John McCarthy coined the term Artificial Intel...,1.000000,0.836618,NaN,1.000
1,What was the significance of the Dartmouth Con...,['The conference was organized by researchers ...,The significance of the Dartmouth Conference i...,The Dartmouth Conference of 1956 is considered...,0.600000,0.687414,1.0,1.000
2,Why did the first AI Winter occur?,['Many AI systems performed well in controlled...,The First AI Winter occurred because researche...,The first AI Winter occurred due to unmet expe...,0.428571,0.846030,1.0,1.000
3,What is machine learning?,['Input Data + Correct Outputs → Learning Algo...,Machine learning is the ability of a model to ...,Machine learning is a branch of AI that enable...,1.000000,0.785267,1.0,0.000
4,What is the difference between supervised and ...,['Supervised Learning 2. Unsupervised Learning...,"Based on the provided context, the difference ...","Supervised learning uses labeled data, while u...",0.600000,0.000000,0.0,1.000
5,What is overfitting in machine learning?,['Overfitting and Underfitting Two common mach...,Overfitting in machine learning occurs when a ...,Overfitting occurs when a model learns the tra...,1.000000,0.798175,1.0,1.000
6,What is backpropagation?,['Adjust weights accordingly. The error signal...,Backpropagation is the process by which the er...,Backpropagation is a training algorithm that u...,0.500000,0.796973,1.0,1.000
7,Why are activation functions important in neur...,['Activation functions introduce non-linearity...,Activation functions are important in neural n...,Activation functions introduce non-linearity i...,1.000000,0.832896,1.0,0.375
8,How do CNNs differ from traditional neural net...,['They are specifically designed for image pro...,"According to the provided context, CNNs differ...",Convolutional Neural Networks use convolution ...,1.000000,0.853351,1.0,1.000
